# Cost-Sensitive Adaptive Demand Forecasting — Full Pipeline

**Sahana Sampath Saraswathy · MSc Data Science**

This notebook runs the project's four pipeline scripts, in order, against the
**corrected** cost model (the annual→weekly carrying-rate fix, and the
z-space adaptive redesign — see `src/cost_model.py` for the full rationale
in the module docstring).

Each stage below corresponds directly to a result shown in
`Examiner_Presentation.pptx`:

| Script | What it establishes | Deck slide |
|---|---|---|
| `01_baseline_comparison.py` | The original discovery: pure ML loses to a naive baseline under trend | Motivates the gate mechanism (Slide 4) |
| `02_gate_diagnostic_trend_sign.py` | Proves the gate fixes that problem | Slide 4 (methodology) |
| `03_comparison_A_and_B_gated_hybrid.py` | Synthetic results — Comparisons A, B, and C (RQ1/RQ2) | Slide 6 |
| `04_real_data_validation.py` | Real H&M validation — pooled + per-category results | Slide 7 |

**Total runtime, all four stages: roughly 20–25 minutes** (01: ~2–3 min, 02: ~4 min, 03: ~4–5 min, 04: ~10–15 min). Run it when you have that window free, not five minutes before you present.

**Before running:** this notebook must live in the `pipeline_corrected/`
root folder (the same folder that contains `scripts/`, `src/`, `data/`,
and `results/`) — every path below is relative to that location. Run the
cells top to bottom, in order; later stages don't strictly require earlier
ones to have run in the same session, but this is the order the project's
own narrative follows, and `04` expects `03`'s output conventions on disk.


In [1]:
git --version

NameError: name 'git' is not defined

## 0. Setup

Confirms you're running from the right folder and that all required
packages are installed. If a package is missing, uncomment the `pip
install` line below and run it once.


In [1]:
import os

NB_DIR = os.getcwd()
SCRIPTS_DIR = os.path.join(NB_DIR, "scripts")
DATA_DIR = os.path.join(NB_DIR, "data", "hm")

assert os.path.isdir(SCRIPTS_DIR), (
    "Run this notebook from the pipeline_corrected root folder "
    "(the folder containing scripts/, src/, data/, results/)."
)
print("Working directory:", NB_DIR)
print("Scripts found:", sorted(os.listdir(SCRIPTS_DIR)))

have_hm_data = os.path.exists(os.path.join(DATA_DIR, "transactions_train.csv")) and \
               os.path.exists(os.path.join(DATA_DIR, "articles.csv"))
print("H&M data present (needed for Stage 4):", have_hm_data)


Working directory: C:\Users\SS SAHANA\Downloads\pipeline_corrected_fix
Scripts found: ['01_baseline_comparison.py', '02_gate_diagnostic_trend_sign.py', '03_comparison_A_and_B_gated_hybrid.py', '04_real_data_validation.py']
H&M data present (needed for Stage 4): True


In [2]:
%pip install -q -r requirements.txt


Note: you may need to restart the kernel to use updated packages.


## 1. Baseline comparison — the original discovery

Compares the pure LightGBM model (no gate, no hybrid) against three
increasingly-sophisticated baselines, on synthetic data, under the same
cost model used everywhere else in the project. This reproduces the
finding that started the whole gating investigation: a plain ML model can
lose 15–24% to a naive rolling-mean-plus-safety-stock baseline when demand
has a trend, because it extrapolates trend into the residual correction
term. Runtime: roughly 2–3 minutes.


In [3]:
%run scripts/01_baseline_comparison.py


Running 50 seeds across 3 products x 4 methods...
Done. Raw results saved to C:\Users\SS SAHANA\Downloads\pipeline_corrected_fix\scripts\..\results\01_baseline_comparison.csv

BASELINE COMPARISON: average weekly £ cost during volatile periods, by method

product_A:
                                               mean         std
method                                                         
4. LightGBM adaptive (pure, no hybrid)   666.112206  537.991663
1. Naive mean + safety stock             674.670288  468.436677
2. Exp. smoothing + safety margin        939.431557  837.450806
3. Naive step-adaptive heuristic        1287.110038  986.493743

product_B:
                                               mean          std
method                                                          
4. LightGBM adaptive (pure, no hybrid)   406.869086   350.557488
1. Naive mean + safety stock             639.100044   403.928629
2. Exp. smoothing + safety margin       1031.954106  1210.345119
3. Naive step

## 2. Gate diagnostic — proving the fix

Runs a controlled 5-condition trend sweep (identical demand level and
volatility, only the trend direction/magnitude varies) twice: once with
the gate disabled (equivalent to `gate_threshold=+inf`), once with it
enabled. Confirms the gate specifically neutralizes the trend-driven harm
found in Stage 1, without needing to know the trend sign in advance — it's
inferred at prediction time from the same rolling trend-slope feature used
everywhere else. Runtime: roughly 4 minutes.


In [4]:
%run scripts/02_gate_diagnostic_trend_sign.py


PART 1: UNGATED hybrid vs naive, across the trend-sign sweep
  strong_negative             : n= 300  mean=+15.97%  CI=[+7.35%, +23.69%]  half-width=8.17pp
  mild_negative               : n= 300  mean=+19.23%  CI=[+13.52%, +24.52%]  half-width=5.50pp
  no_trend                    : n= 300  mean=+18.94%  CI=[+11.44%, +25.14%]  half-width=6.85pp
  mild_positive               : n= 300  mean=+20.80%  CI=[+15.14%, +25.84%]  half-width=5.35pp
  strong_positive             : n= 300  mean=+21.65%  CI=[+16.48%, +26.42%]  half-width=4.97pp

REGRESSION: ungated hybrid-vs-naive gap against trend_per_week
Slope: 2589.7  (% improvement per unit of trend_per_week)
R-squared: 0.881   p-value: 0.0181
-> SIGNIFICANT relationship between trend sign and hybrid performance

PART 2: GATED hybrid vs naive, same sweep (threshold=4.6, pre-committed from no-trend training data)
  strong_negative             : n= 300  mean=+20.25%  CI=[+16.05%, +24.59%]  half-width=4.27pp
  mild_negative               : n= 300  m

## 3. Synthetic results — Comparisons A, B, and C

The main synthetic-data result. For each of the three products (Steady
Grower / Declining Item / Fast Grower), this runs:

- **Comparison A** — adaptive gated hybrid vs. a naive safety-stock
  baseline (no ML at all)
- **Comparison B** — adaptive vs. fixed target quantile — **this is the
  actual thesis (RQ1/RQ2)**
- **Comparison C** — adaptive vs. a standard-MSE-trained point forecast
  (closes the other half of RQ2)

Each comparison uses a pre-committed stopping rule (batches of 50 seeds,
stops once the bootstrap CI half-width drops below target, capped at
400–600 seeds) rather than a fixed single run, so the sample size you see
in the output is not hand-picked. Runtime: roughly 4–5 minutes.


In [5]:
%run scripts/03_comparison_A_and_B_gated_hybrid.py


COMPARISON A: gated hybrid (adaptive) vs. naive baseline
  product_A                   : n= 400  mean=+18.57%  CI=[+14.71%, +22.07%]  half-width=3.68pp
  product_B                   : n= 400  mean=+21.37%  CI=[+18.25%, +24.47%]  half-width=3.11pp
  product_C                   : n= 400  mean=+18.88%  CI=[+15.90%, +21.88%]  half-width=2.99pp

COMPARISON B: adaptive vs. fixed target (THE THESIS)
  product_A                   : n= 600  mean=+0.97%  CI=[-0.20%, +2.06%]  half-width=1.13pp
  product_B                   : n= 600  mean=+4.89%  CI=[+3.65%, +6.20%]  half-width=1.27pp
  product_C                   : n= 600  mean=+0.41%  CI=[-1.01%, +1.49%]  half-width=1.25pp

COMPARISON C: adaptive vs. standard MSE (closes RQ2)
  product_A                   : n= 400  mean=+93.19%  CI=[+92.65%, +93.74%]  half-width=0.55pp
  product_B                   : n= 400  mean=+91.94%  CI=[+91.31%, +92.58%]  half-width=0.63pp
  product_C                   : n= 400  mean=+93.67%  CI=[+93.14%, +94.18%]  half-wi

## 4. Real-data validation — H&M

Requires `transactions_train.csv` and `articles.csv` in
`data/hm/` (checked in Stage 0 above). Runs the same walk-forward,
cluster-bootstrapped validation on real H&M transaction data, across the
top-N articles by transaction volume. **Pass `1500` as the argument below
to match the numbers in the deck exactly** — the default (300) will give a
smaller, noisier sample. Runtime: roughly 10–15 minutes at N=1500; the
script prints per-article progress as it goes, so it's not stuck if it
looks quiet for a few seconds at a time.


In [6]:
%run scripts/04_real_data_validation.py 1500


Schema check passed: transactions_train.csv has ['t_dat', 'customer_id', 'article_id', 'price', 'sales_channel_id']
                     articles.csv has 25 columns (checked: ['article_id', 'product_type_name', 'index_name'])

Built weekly panels for 1284 articles (target was 1500).

Article 108775015: 93 weeks of history, empirical CV range [0.16, 3.46], median CV=0.73
  Fold 0: train=63wk test=10wk  q*=0.998  gate_threshold=-0.02
    naive=£0.00  fixed=£0.00  adaptive=£0.00  mse=£0.02
  Fold 1: train=73wk test=10wk  q*=0.998  gate_threshold=0.00
    naive=£0.00  fixed=£0.00  adaptive=£0.00  mse=£0.00
  Fold 2: train=83wk test=10wk  q*=0.998  gate_threshold=0.00
    naive=£0.00  fixed=£0.00  adaptive=£0.00  mse=£0.00

Article 108775044: 101 weeks of history, empirical CV range [0.18, 1.17], median CV=0.56
  Fold 0: train=71wk test=10wk  q*=0.998  gate_threshold=1.85
    naive=£0.01  fixed=£0.00  adaptive=£0.00  mse=£0.00
  Fold 1: train=81wk test=10wk  q*=0.998  gate_threshold=1.09
  

## Where the results land

Every stage above also writes its output to `results/` as it runs, so the
numbers survive independently of this notebook:

- `results/01_baseline_comparison.csv`
- `results/03_comparison_A_and_B_summary.csv`
- `results/04_real_data_folds_N1500.csv`

These are the same files the deck's numbers were pulled from — if you
re-run this notebook fresh, you should see the same figures reappear
(seeds are fixed, so the synthetic results are bit-for-bit reproducible;
the real-data results depend only on the fixed walk-forward split, not on
any randomness, so they reproduce exactly too).
